# Bronze to Silver — Incremental

This notebook is the incremental version of the batch Bronze-to-Silver pipeline.

**Bronze files → Auto Loader → `parse_aemo()` → append-only Delta candidates → incrementally refreshed materialized Silver views**

The business logic is unchanged:

- actual demand prefers Current → Daily → Monthly;
- forecasts retain `(forecast_time, time, demand)`;
- the final Silver table names and schemas remain unchanged.

The incremental candidate tables use new names so the first Auto Loader run does not append the historical files on top of candidate rows previously created by the batch notebook.

Only infrastructure changes. Parsing, filters, source priority and final output schemas remain the same, making the transition from batch to incremental easy to compare.


In [ ]:
from pyspark.sql import functions as F


## Parse the AEMO files

The parser keeps the same AEMO logic as the batch implementation.

The batch notebook used one `spark.read.text(path)` DataFrame for every matching file. Here that work is split in two: a small batch read finds the `I` row and resolves the AEMO column positions; Auto Loader creates the streaming plan for matching `D` rows.

The row pattern remains explicit: `DISPATCH,REGIONSUM` contains actual demand, while `P5MIN,REGIONSOLUTION` contains AEMO forecasts. The transformations are still lazy: this function defines how rows will be processed; the later streaming write performs the work.


In [ ]:
def parse_aemo(
    path,
    row_pattern,
    time_column,
    forecast_time_column=None,
    filter_runno=False,
):
    # Unchanged idea: read one I row to obtain this section's column positions.
    header_raw = spark.read.text(f"{path}/*.CSV")

    header_row = (
        header_raw
        .filter(F.col("value").startswith(f"I,{row_pattern},"))
        .select(F.split("value", ",").alias("fields"))
        .first()
    )

    if header_row is None:
        raise ValueError(f"No I row found for {row_pattern} in {path}")

    header = header_row["fields"]
    time_i = header.index(time_column)
    region_i = header.index("REGIONID")
    intervention_i = header.index("INTERVENTION")
    demand_i = header.index("TOTALDEMAND")

    # Batch version: raw = spark.read.text(path) read every matching file again.
    # Auto Loader replaces that read and uses a checkpoint to discover new files.
    raw = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "text")
        .option("pathGlobFilter", "*.CSV")
        .load(path)
    )

    # The parsing and VIC1 filters below are unchanged from the batch version.
    rows = (
        raw
        .filter(F.col("value").startswith(f"D,{row_pattern},"))
        .withColumn("fields", F.split("value", ","))
        .filter(F.col("fields")[region_i] == "VIC1")
        .filter(F.col("fields")[intervention_i].cast("int") == 0)
    )

    if filter_runno:
        runno_i = header.index("RUNNO")
        rows = rows.filter(F.col("fields")[runno_i].cast("int") == 1)

    time = F.to_timestamp(
        F.regexp_replace(F.col("fields")[time_i], '"', ""),
        "yyyy/MM/dd HH:mm:ss",
    ).alias("time")

    demand = (
        F.regexp_replace(F.col("fields")[demand_i], '"', "")
        .cast("double")
        .alias("demand")
    )

    if forecast_time_column is None:
        return rows.select(time, demand)

    forecast_time_i = header.index(forecast_time_column)
    forecast_time = F.to_timestamp(
        F.regexp_replace(F.col("fields")[forecast_time_i], '"', ""),
        "yyyy/MM/dd HH:mm:ss",
    ).alias("forecast_time")

    return rows.select(forecast_time, time, demand)


## Actual demand

Monthly, Daily and Current contain the same observed VIC1 demand delivered through different AEMO layers. The DataFrame definitions remain separate because each source needs its own file-discovery checkpoint.

Priority preserves the same batch rule: Current, then Daily, then Monthly. The only visible path change is that Auto Loader receives a folder, while the batch read received a `*.CSV` pattern.


In [ ]:
bronze_path = "/Volumes/workspace/default/aemo_mlops_volume/bronze"

monthly = (
    parse_aemo(
        f"{bronze_path}/monthly_uncompressed",
        row_pattern="DISPATCH,REGIONSUM",
        time_column="SETTLEMENTDATE",
        filter_runno=True,
    )
    .withColumn("priority", F.lit(1))
)

daily = (
    parse_aemo(
        f"{bronze_path}/daily_uncompressed",
        row_pattern="DISPATCH,REGIONSUM",
        time_column="SETTLEMENTDATE",
        filter_runno=True,
    )
    .withColumn("priority", F.lit(2))
)

current = (
    parse_aemo(
        f"{bronze_path}/current_uncompressed",
        row_pattern="DISPATCH,REGIONSUM",
        time_column="SETTLEMENTDATE",
        filter_runno=True,
    )
    .withColumn("priority", F.lit(3))
)


## AEMO demand forecast

P5MIN produces a new forecast run every five minutes. `forecast_time` is when AEMO produced the run; `time` is the future interval being predicted. The pair remains the forecast key. Archive and Current stay separate because their new files arrive independently.


In [ ]:
forecast_archive = parse_aemo(
    f"{bronze_path}/forecast/archive_uncompressed",
    row_pattern="P5MIN,REGIONSOLUTION",
    time_column="INTERVAL_DATETIME",
    forecast_time_column="RUN_DATETIME",
)

forecast_current = parse_aemo(
    f"{bronze_path}/forecast/current_uncompressed",
    row_pattern="P5MIN,REGIONSOLUTION",
    time_column="INTERVAL_DATETIME",
    forecast_time_column="RUN_DATETIME",
)


## Create incremental candidate tables

The batch notebook used temporary views followed by `CREATE OR REPLACE TABLE ... AS SELECT`. That rebuilt candidate history from every Bronze file. Temporary views are no longer needed: the streaming DataFrames write directly to persistent candidate tables.

These candidate tables use new names because Auto Loader starts with empty checkpoints. On its first run it must ingest the historical Bronze files once; new tables prevent those rows from being appended on top of candidate history created by the batch notebook.

Duplicates remain intentional and are resolved downstream using the same rules as before. Row tracking records which candidate rows changed so the materialized views can update only affected results.


In [ ]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.demand_vic_5min_incremental_candidates (
    time TIMESTAMP,
    demand DOUBLE,
    priority INT
)
USING DELTA
TBLPROPERTIES ('delta.enableRowTracking' = 'true')


In [ ]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.aemo_demand_forecast_vic_5min_incremental_candidates (
    forecast_time TIMESTAMP,
    time TIMESTAMP,
    demand DOUBLE
)
USING DELTA
TBLPROPERTIES ('delta.enableRowTracking' = 'true')


## Append only new files

The batch version needed no checkpoints because it deliberately reread everything. Auto Loader needs one checkpoint per source to remember which files have already contributed rows. The paths below are new and have never been run.

`availableNow=True` processes all currently unprocessed files and then stops, so this remains a finite scheduled task rather than a permanently running stream. On the first run it loads the existing Bronze history; later runs append only rows from new files.

A candidate table and its checkpoints form one state. If this test is ever reset, clear both together; deleting only a checkpoint would make Auto Loader treat historical files as new and append them again.


In [ ]:
def append_new_files(df, table, checkpoint):
    # Batch version: CREATE OR REPLACE rewrote the complete candidate table.
    # Incremental version: append rows only from files absent from this checkpoint.
    query = (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint)
        .trigger(availableNow=True)
        .toTable(table)
    )

    # availableNow stops only after the current backlog has been processed.
    query.awaitTermination()


append_new_files(
    monthly,
    "workspace.default.demand_vic_5min_incremental_candidates",
    "/Volumes/workspace/default/aemo_mlops_volume/checkpoints/demand_actual_monthly",
)

append_new_files(
    daily,
    "workspace.default.demand_vic_5min_incremental_candidates",
    "/Volumes/workspace/default/aemo_mlops_volume/checkpoints/demand_actual_daily",
)

append_new_files(
    current,
    "workspace.default.demand_vic_5min_incremental_candidates",
    "/Volumes/workspace/default/aemo_mlops_volume/checkpoints/demand_actual_current",
)

append_new_files(
    forecast_archive,
    "workspace.default.aemo_demand_forecast_vic_5min_incremental_candidates",
    "/Volumes/workspace/default/aemo_mlops_volume/checkpoints/demand_forecast_archive",
)

append_new_files(
    forecast_current,
    "workspace.default.aemo_demand_forecast_vic_5min_incremental_candidates",
    "/Volumes/workspace/default/aemo_mlops_volume/checkpoints/demand_forecast_current",
)


## Validate that the Silver queries can be incremental

The batch queries could always rebuild their outputs. The incremental version first uses `EXPLAIN CREATE MATERIALIZED VIEW` to check whether Databricks can update the same result from source-table changes. This validates query structure; the event log later validates what happened during execution.

The expected result includes:

`The Materialized View can be incrementally refreshed.`


In [ ]:
%sql
EXPLAIN CREATE MATERIALIZED VIEW test_demand_view
AS

SELECT
    time,
    demand
FROM workspace.default.demand_vic_5min_incremental_candidates

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY time
    ORDER BY priority DESC
) = 1


In [ ]:
%sql
EXPLAIN CREATE MATERIALIZED VIEW test_forecast_view
AS

SELECT
    forecast_time,
    time,
    demand
FROM workspace.default.aemo_demand_forecast_vic_5min_incremental_candidates
GROUP BY
    forecast_time,
    time,
    demand


## One-time transition from the batch Silver tables

The batch notebook created the final Silver outputs with `CREATE OR REPLACE TABLE`. They must be removed once before the same names can be created as materialized views. This changes the object that maintains the result, not the table names seen by downstream modelling.

This block only drops them while they are ordinary managed or external tables. Once they are materialized views, later runs leave them unchanged.


In [ ]:
%sql
BEGIN
    IF EXISTS (
        SELECT 1
        FROM workspace.information_schema.tables
        WHERE table_schema = 'default'
          AND table_name = 'demand_vic_5min'
          AND table_type IN ('MANAGED', 'EXTERNAL')
    ) THEN
        DROP TABLE workspace.default.demand_vic_5min;
    END IF;

    IF EXISTS (
        SELECT 1
        FROM workspace.information_schema.tables
        WHERE table_schema = 'default'
          AND table_name = 'aemo_demand_forecast_vic_5min'
          AND table_type IN ('MANAGED', 'EXTERNAL')
    ) THEN
        DROP TABLE workspace.default.aemo_demand_forecast_vic_5min;
    END IF;
END


## Build the actual Silver materialized view

The batch version used `CREATE OR REPLACE TABLE` plus a `ROW_NUMBER()` CTE. The incremental version keeps the same ranking rule but expresses the filter with `QUALIFY`, which Databricks can incrementally maintain against the row-tracked candidate table.

`INCREMENTAL STRICT` makes the incremental behaviour testable: after initial creation, a refresh fails rather than silently falling back to a full recomputation if Databricks cannot use an incremental plan.


In [ ]:
%sql
CREATE MATERIALIZED VIEW IF NOT EXISTS workspace.default.demand_vic_5min
REFRESH POLICY INCREMENTAL STRICT
AS

SELECT
    time,
    demand
FROM workspace.default.demand_vic_5min_incremental_candidates

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY time
    ORDER BY priority DESC
) = 1


In [ ]:
%sql
REFRESH MATERIALIZED VIEW workspace.default.demand_vic_5min


## Build the forecast Silver materialized view

The batch version used `SELECT DISTINCT forecast_time, time, demand`. Grouping by the same three columns returns the same rows while using an explicitly incrementalizable aggregation. No source precedence is introduced for forecasts.


In [ ]:
%sql
CREATE MATERIALIZED VIEW IF NOT EXISTS workspace.default.aemo_demand_forecast_vic_5min
REFRESH POLICY INCREMENTAL STRICT
AS

SELECT
    forecast_time,
    time,
    demand
FROM workspace.default.aemo_demand_forecast_vic_5min_incremental_candidates
GROUP BY
    forecast_time,
    time,
    demand


In [ ]:
%sql
REFRESH MATERIALIZED VIEW workspace.default.aemo_demand_forecast_vic_5min


## Validate the Silver outputs

These are the same output checks used by the batch baseline. The implementation changed, but the contracts did not: actual demand has one row per `time`, forecasts have one row per `(forecast_time, time)`, and required values cannot be null. A healthy run returns two summary rows with `invalid_rows = 0`, followed by an empty duplicate-key result.


In [ ]:
%sql
SELECT
    'actual demand' AS dataset,
    COUNT(*) AS rows,
    MIN(time) AS first_target_time,
    MAX(time) AS last_target_time,
    COUNT_IF(time IS NULL OR demand IS NULL) AS invalid_rows
FROM workspace.default.demand_vic_5min

UNION ALL

SELECT
    'AEMO forecast' AS dataset,
    COUNT(*) AS rows,
    MIN(time) AS first_target_time,
    MAX(time) AS last_target_time,
    COUNT_IF(forecast_time IS NULL OR time IS NULL OR demand IS NULL) AS invalid_rows
FROM workspace.default.aemo_demand_forecast_vic_5min


In [ ]:
%sql
SELECT 'actual demand' AS dataset, time AS first_key, CAST(NULL AS TIMESTAMP) AS second_key, COUNT(*) AS rows
FROM workspace.default.demand_vic_5min
GROUP BY time
HAVING COUNT(*) > 1

UNION ALL

SELECT 'AEMO forecast' AS dataset, forecast_time AS first_key, time AS second_key, COUNT(*) AS rows
FROM workspace.default.aemo_demand_forecast_vic_5min
GROUP BY forecast_time, time
HAVING COUNT(*) > 1


## Validate conflicting values

Exact duplicates are harmless. The first query checks overlapping actual observations before priority is applied; the second checks forecast keys after exact deduplication.


In [ ]:
%sql
SELECT
    time,
    SORT_ARRAY(COLLECT_SET(demand)) AS conflicting_values
FROM workspace.default.demand_vic_5min_incremental_candidates
GROUP BY time
HAVING COUNT(DISTINCT demand) > 1
ORDER BY time


In [ ]:
%sql
SELECT
    forecast_time,
    time,
    SORT_ARRAY(COLLECT_SET(demand)) AS conflicting_values
FROM workspace.default.aemo_demand_forecast_vic_5min
GROUP BY forecast_time, time
HAVING COUNT(DISTINCT demand) > 1
ORDER BY forecast_time, time


## Validate the refresh type

`EXPLAIN` proves that the query can be incrementalized. The event log shows which technique Databricks actually used during the refresh.

An incremental refresh reports techniques such as `WINDOW_FUNCTION`, `ROW_BASED`, `APPEND_ONLY` or `GROUP_AGGREGATE`. `FULL_RECOMPUTE` means the refresh was not incremental.


In [ ]:
%sql
SELECT
    timestamp,
    message
FROM event_log(
    TABLE(workspace.default.demand_vic_5min)
)
WHERE event_type = 'planning_information'
ORDER BY timestamp DESC
LIMIT 5


In [ ]:
%sql
SELECT
    timestamp,
    message
FROM event_log(
    TABLE(workspace.default.aemo_demand_forecast_vic_5min)
)
WHERE event_type = 'planning_information'
ORDER BY timestamp DESC
LIMIT 5


## Test the incremental behaviour

1. Run the notebook once: the new checkpoints load the existing Bronze history.
2. Record both candidate-table row counts and run it again without adding files: the counts should not change.
3. Add one new Current file and run it again: only rows from that file should be appended.
4. Confirm the Silver schemas and uniqueness checks still pass, then inspect the event logs for an incremental technique rather than `FULL_RECOMPUTE`.

This test isolates the intended improvement: the same data contracts are maintained without rereading and rebuilding the complete history on every run.
